# DeepSeek V3 on Amazon Bedrock

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

DeepSeek's V3 models are strong at multi-step reasoning and mathematics. `deepseek.v3.2`
is the current generation and the one this notebook calls; `deepseek.v3.1` is the
previous one, which is useful for checking a migration.

| Model | `bedrock-mantle` ID | `bedrock-runtime` ID | Notes |
|---|---|---|---|
| DeepSeek V3.2 | `deepseek.v3.2` | `deepseek.v3.2` | Current generation |
| DeepSeek V3.1 | `deepseek.v3.1` | — | Previous generation, `bedrock-mantle` only |

IDs as of September 2026, in `us-east-1`. Which Regions carry each model changes as
models launch, so the setup cell checks your Region.

On `bedrock-mantle` these models serve the OpenAI-compatible **Chat Completions** API at
the bare `/v1` path. On `bedrock-runtime`, V3.2 serves **Converse** under the same ID
(section 7).

**Covered:** first call, streaming, multi-turn, reasoning, tool use, structured output,
and Converse on `bedrock-runtime`.

## Setup

Install the pinned requirements first (`pip install -r ../requirements.txt`, Python 3.11
or later). The notebook calls Bedrock with the OpenAI SDK and boto3. From
`_shared/bedrock.py`, this repository's own helper module, it takes an availability
check, a lenient JSON parser, and a repair for a cut-off tool-call arguments
string.

Endpoints, authentication and URL paths are explained in
[`00-foundations/01`](../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb).

In [1]:
import json
import sys

import boto3
from aws_bedrock_token_generator import provide_token
from openai import OpenAI

sys.path.insert(0, "../_shared")
from bedrock import endpoints_for, parse_json_lenient, repair_tool_arguments

REGION = "us-east-1"
V32 = "deepseek.v3.2"
V31 = "deepseek.v3.1"

# bedrock-mantle: the OpenAI SDK, with a short-term Bedrock API key minted from your
# AWS credentials. Mint it when you build the client, because the key expires.
client = OpenAI(
    base_url=f"https://bedrock-mantle.{REGION}.api.aws/v1",
    api_key=provide_token(region=REGION),
    timeout=120,
)

# bedrock-runtime: boto3 signs each request with SigV4, so there is no key to mint.
runtime = boto3.client("bedrock-runtime", region_name=REGION)

for model in (V32, V31,):
    served = [name for name, ok in endpoints_for(model, REGION).items() if ok]
    print(f"{model:15} served by: {', '.join(served) or 'neither endpoint in ' + REGION}")

deepseek.v3.2   served by: mantle, runtime
deepseek.v3.1   served by: mantle


## 1. First call

A single request with a short question. If the model spends the whole `max_tokens`
budget before it answers, `content` is `None` and `finish_reason` is `length`, so the
cell prints both.

In [2]:
response = client.chat.completions.create(
    model=V32,
    messages=[{"role": "user", "content": "What is the capital of Australia? Answer in one word."}],
    max_tokens=1000,
)
choice = response.choices[0]
answer = (choice.message.content or "").strip()
print(answer, "| Canberra:", "canberra" in answer.lower())
print(f"finish_reason={choice.finish_reason}  total_tokens={response.usage.total_tokens}")

Canberra | Canberra: True
finish_reason=stop  total_tokens=19


## 2. Streaming

Set `stream=True` and read `delta.content` from each chunk. Check that a chunk has
choices before indexing it, and read `finish_reason` from the last one: `stop` means the
answer is complete.

In [3]:
stream = client.chat.completions.create(
    model=V32,
    messages=[{"role": "user", "content": "Count from 1 to 10, separated by commas."}],
    max_tokens=1000,
    stream=True,
)
finish_reason = None
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
    if chunk.choices and chunk.choices[0].finish_reason:
        finish_reason = chunk.choices[0].finish_reason
print(f"\nfinish_reason={finish_reason}")

1,

2

, 3, 4

, 5, 6, 7, 8, 

9, 10


finish_reason=stop


## 3. Multi-turn

Chat Completions keeps no conversation state on the server, so each request carries the
whole history. Append the assistant's reply before the next user turn.

In [4]:
messages = [
    {"role": "system", "content": "You are a concise assistant."},
    {"role": "user", "content": "My favourite colour is teal. Reply with one word: noted."},
]
first = client.chat.completions.create(model=V32, messages=messages, max_tokens=1000)
messages.append({"role": "assistant", "content": first.choices[0].message.content})
messages.append({"role": "user", "content": "What is my favourite colour? One word."})
second = client.chat.completions.create(model=V32, messages=messages, max_tokens=1000)

print("turn 1:", (first.choices[0].message.content or "").strip())
print("turn 2:", (second.choices[0].message.content or "").strip())

turn 1: noted
turn 2: teal


## 4. Reasoning

Set `reasoning_effort` to `low`, `medium` or `high` and V3.2 reasons before it answers.
The reasoning comes back in `choices[0].message.reasoning`, a field the OpenAI schema
does not define. The SDK still exposes it as an attribute, but code written against the
published types will not see it. Reasoning is billed as output and spends the budget
before the answer starts, so give it a larger `max_tokens`.

In [5]:
response = client.chat.completions.create(
    model=V32,
    messages=[{"role": "user", "content": 'A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? Answer with the amount only.'}],
    reasoning_effort="high",
    max_tokens=4000,
)
message = response.choices[0].message
trace = getattr(message, "reasoning", None) or ""
answer = (message.content or "").strip()
print("answer   :", answer, "| 0.05 present:", "0.05" in answer)
print(f"reasoning: {len(trace)} characters")
print(f"finish_reason={response.choices[0].finish_reason}  "
      f"completion_tokens={response.usage.completion_tokens}")

answer   : The ball costs $0.05.

Let the cost of the ball be \(x\) dollars. Then the bat costs \(x + 1.00\) dollars. The total cost is $1.10, so:
\[
x + (x + 1.00) = 1.10
\]
\[
2x + 1.00 = 1.10
\]
\[
2x = 0.10
\]
\[
x = 0.05
\]

\boxed{0.05} | 0.05 present: True
reasoning: 1527 characters
finish_reason=stop  completion_tokens=573


## 5. Tool use

A tool loop sends the tool definitions, runs each call the model makes, returns the
results, and repeats until the model answers in text. Decide what to run from the raw
arguments string, parsed strictly: `finish_reason == "tool_calls"` says the model wants a
tool and says nothing about whether the arguments arrived complete, and a lenient parse
closes a cut-off value into a call that looks usable. On V3.2 a short budget did not
produce a cut-off arguments string in 40 trials, so the strict parse here guards against
something this model has not been seen to do; the copy of the turn echoed back is still
repaired, because an arguments string cut mid-value is refused. Things that differ has
both measurements.

V3.2 writes its own function-call markup (`<｜DSML｜function_calls…`) into `content`,
after a sentence or two of preamble, alongside a usable tool call: 6 of 6 trials with a
1,500-token budget came back with `finish_reason` `tool_calls`, valid arguments, and that
markup in `content`. The markup is not a failure on its own, and the text of a turn that
carries a tool call is not the answer. What a short budget returns instead is a turn with
`finish_reason` `length` and no tool call at all, carrying the preamble and sometimes the
markup, and the loop treats that as a failed turn and asks again rather than reading the
preamble as the answer.

In [6]:
def lookup_inventory(sku: str) -> dict:
    """Stand-in for a real inventory service."""
    stock = {"A-100": 42, "B-200": 0}
    if sku.upper() not in stock:
        return {"sku": sku, "error": "unknown SKU"}
    return {"sku": sku, "quantity": stock[sku.upper()]}


TOOLS = [{
    "type": "function",
    "function": {
        "name": "lookup_inventory",
        "description": "Current stock level for a SKU.",
        "parameters": {
            "type": "object",
            "properties": {"sku": {"type": "string", "description": "SKU code, e.g. A-100"}},
            "required": ["sku"],
        },
    },
}]
IMPLEMENTATIONS = {"lookup_inventory": lookup_inventory}


def run_tool(name: str, args: dict) -> dict:
    """Refuse a call that names an unknown tool or does not match the tool's parameters."""
    spec = next((t["function"] for t in TOOLS if t["function"]["name"] == name), None)
    if spec is None:
        return {"error": f"unknown tool {name}"}
    params = spec["parameters"]
    missing = [k for k in params["required"] if k not in args]
    unknown = [k for k in args if k not in params["properties"]]
    wrong_type = [k for k in args if k in params["properties"] and params["properties"][k].get("type") == "string"
                  and not isinstance(args[k], str)]
    if missing or unknown or wrong_type:
        return {"error": f"bad arguments: missing {missing}, unknown {unknown}, not a string {wrong_type}"}
    return IMPLEMENTATIONS[name](**args)

In [7]:
messages = [{"role": "user", "content": "How many units of SKU A-100 do we have?"}]
for _ in range(5):  # bounded, so a model that keeps calling tools cannot loop forever
    response = client.chat.completions.create(
        model=V32, messages=messages, tools=TOOLS, max_tokens=1500
    )
    choice = response.choices[0]
    message = choice.message
    if not message.tool_calls:
        # V3.2 echoes its function-call markup into content alongside a usable call, so
        # the markup means a failed turn only when no call came with it, and a budget
        # that ran out mid-turn arrives as text rather than as a cut-off call.
        if choice.finish_reason == "length" or "<｜DSML｜" in (message.content or ""):
            continue
        break
    raw_args = [call.function.arguments for call in message.tool_calls]
    # The echoed turn has to carry valid JSON or the next request is refused, so repair the
    # copy that goes back. What actually runs is decided from the raw string, because the
    # repair can close a cut-off value into arguments that look complete.
    turn = message.model_dump(exclude_none=True)
    for call, raw in zip(turn["tool_calls"], raw_args):
        call["function"]["arguments"] = repair_tool_arguments(raw)
    messages.append(turn)
    for call, raw in zip(turn["tool_calls"], raw_args):
        name = call["function"]["name"]
        try:
            args = json.loads(raw)
        except json.JSONDecodeError as exc:
            args, result = {}, {"error": f"arguments cut off ({exc.msg}); send them again"}
        else:
            result = run_tool(name, args)
        print(f"tool call: {name}({args}) -> {result}")
        messages.append({"role": "tool", "tool_call_id": call["id"], "content": json.dumps(result)})
else:
    raise RuntimeError("no answer after 5 rounds of tool calls")

answer = (message.content or "").strip()
print("answer:", answer)
# 42 is the quantity the stand-in tool reports for A-100, so the answer must carry it.
print("42 units reported:", "42" in answer)

tool call: lookup_inventory({'sku': 'A-100'}) -> {'sku': 'A-100', 'quantity': 42}


answer: Based on the current inventory data, you have **42 units** of SKU A-100 in stock.
42 units reported: True


## 6. Structured output

`response_format` with a JSON Schema asks the model for output in that shape. Check
`finish_reason` first, because a cut-off answer is cut-off JSON. Parse leniently,
because some models add text after a valid object. A schema controls the shape of the
output and has no say over the values in it, so the cell checks each value against the
source text.

In [8]:
SCHEMA = {
    "type": "object",
    "properties": {
        "invoice_id": {"type": "string"},
        "vendor": {"type": "string"},
        "total": {"type": "number"},
        "currency": {"type": "string"},
    },
    "required": ["invoice_id", "vendor", "total", "currency"],
    "additionalProperties": False,
}
SOURCE = "Invoice: INV-1042\nVendor: Acme Pty Ltd\nAmount due: AUD 1,280.50\nDue date: 30 October 2026"
EXPECTED = {"invoice_id": "INV-1042", "vendor": "Acme Pty Ltd", "total": 1280.5, "currency": "AUD"}

response = client.chat.completions.create(
    model=V32,
    messages=[{"role": "user", "content": (
        "Extract invoice_id, vendor, total and currency from this invoice as JSON. Write the "
        f"total as a number without a thousands separator, so 2,345.60 becomes 2345.6.\n\n{SOURCE}")}],
    response_format={"type": "json_schema", "json_schema": {"name": "invoice", "strict": True, "schema": SCHEMA}},
    max_tokens=1500,
)
choice = response.choices[0]
if choice.finish_reason != "stop":
    raise RuntimeError(f"incomplete output (finish_reason={choice.finish_reason}); raise max_tokens")

invoice = parse_json_lenient(choice.message.content)
print(json.dumps(invoice, indent=2))
wrong = {k: invoice.get(k) for k, v in EXPECTED.items() if invoice.get(k) != v}
print("values match the source" if not wrong else f"values that do NOT match the source: {wrong}")

{
  "invoice_id": "INV-1042",
  "vendor": "Acme Pty Ltd",
  "total": 1280.5,
  "currency": "AUD"
}
values match the source


## 7. On `bedrock-runtime`: Converse

V3.2 is on `bedrock-runtime` under the same ID. Converse has one request shape for every
provider: `content` is a list of blocks, and settings such as the token limit go in
`inferenceConfig`. An option only one provider understands, such as `reasoning_effort`,
goes in `additionalModelRequestFields`, which Converse passes to the model unchecked.
Reasoning arrives as a `reasoningContent` block ahead of the text, so read blocks by
type instead of taking `content[0]`. The Converse tool loop is in
[`00-foundations/04`](../00-foundations/04-bedrock-runtime-converse-and-profiles.ipynb).

In [9]:
response = runtime.converse(
    modelId="deepseek.v3.2",
    messages=[{"role": "user", "content": [{"text": "What is the capital of Australia? Answer in one word."}]}],
    inferenceConfig={"maxTokens": 2000},
    additionalModelRequestFields={"reasoning_effort": "high"},
)
blocks = response["output"]["message"]["content"]
text = "".join(b["text"] for b in blocks if "text" in b)
print("blocks    :", [next(iter(b)) for b in blocks])
print("answer    :", text.strip(), "| Canberra:", "canberra" in text.lower())
trace = "".join(b["reasoningContent"].get("reasoningText", {}).get("text", "") for b in blocks if "reasoningContent" in b)
print("reasoning :", len(trace), "characters")
print("stopReason:", response["stopReason"])

blocks    : ['reasoningContent', 'text']
answer    : Canberra | Canberra: True
reasoning : 615 characters
stopReason: end_turn


## Things that differ

What trips people up on this family, and what to do about it.

| Behaviour | What you see | What to do |
|---|---|---|
| Responses API | `POST /v1/responses` returns 400 `The model '…' does not support the '/v1/responses' API` | Use Chat Completions. |
| Path prefix | On `bedrock-mantle`, `/openai/v1/chat/completions` returns 400 ``model `…` isn't supported on this route``. On `bedrock-runtime` that path answers 200 for `deepseek.v3.2` under its bare ID, and 400 `The provided model identifier is invalid.` for `deepseek.v3.1`, which the runtime catalogue does not carry (October 2026) | Use the bare `/v1` path on `bedrock-mantle`. The prefix is not interchangeable between the endpoints: `bedrock-runtime` has an `/openai/v1` route of its own that serves DeepSeek V3.2, so the mantle 400 describes that route rather than the models. |
| Conversation state | No `previous_response_id` | Send the full `messages` list on every turn. |
| Empty answer | HTTP 200, `content` is `None`, `finish_reason` is `length` | The model spent `max_tokens` before answering, often on reasoning. Raise the budget and check `finish_reason` before using the text. |
| Structured output values | Valid JSON that holds a wrong value | A schema only controls the shape. Check the values you depend on. |
| Rate limits | 429 under load | Throttling is token-based and there is no requests-per-minute quota. Retry with backoff ([`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)). |
| Metrics | Nothing under `AWS/Bedrock` in CloudWatch | Mantle metrics are in the `AWS/BedrockMantle` namespace. |
| Function-call markup in `content` | `content` holds a sentence of preamble and then `<｜DSML｜function_calls`, alongside a usable tool call with valid arguments, in 6 of 6 trials with a 1,500-token budget (October 2026) | Normal on V3.2, not a failure. Act on `tool_calls`, and do not read `content` as the answer on a turn that carries a call. The markup means a failed turn only when no tool call came with it, and a check on the first characters of `content` misses it because the preamble comes first (section 5). |
| Cut-off tool call | A short budget does not arrive as cut-off arguments. In 40 trials on `deepseek.v3.2`, `max_tokens` 40 to 480, with a tool taking a path and a 60-word body, 24 returned `finish_reason` `length` and no `tool_calls` at all, carrying a preamble or the markup above in `content`; the other 16 carried arguments that parsed strictly, 4 of them with `completion_tokens` equal to the cap. None arrived as an arguments string that failed a strict parse. An arguments string cut mid-value is still refused when echoed back: 400 `validation_error` carrying `Unterminated string starting at: line 1 column 71 (char 70)`, where the repaired copy answers 200 (`us-east-1`, October 2026) | Treat a `length` turn as failed and ask again rather than reading its text as the answer. Decide what to run from a strict parse of the raw arguments, and repair only the copy you echo back (section 5). |
| V3.1 on `bedrock-runtime` | Not in the `bedrock-runtime` catalogue | Call V3.1 on `bedrock-mantle`, or move to V3.2. |

## Next

- Other families on Chat Completions:
  [`04-qwen/01`](../04-qwen/01-qwen3-core-and-tools.ipynb),
  [`06-zai-glm/01`](../06-zai-glm/01-glm-family.ipynb),
  [`09-minimax/01`](../09-minimax/01-minimax-m2.ipynb)
- Choosing a model and API across families:
  [`99-cross-cutting/01`](../99-cross-cutting/01-choosing-a-model-and-api.ipynb)
- Quotas, retries and latency:
  [`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)
- Production hardening:
  [`99-cross-cutting/03`](../99-cross-cutting/03-production-hardening.ipynb)